# Cleaning the CEPII Gravity Data and Building Trade-/GDP-Weighted Exposure

This notebook builds the **trade- and GDP-weighted exposure variables** flagged as
step 5 of §6.2 in
[`docs/developing-country-trade-productivity.md`](../docs/developing-country-trade-productivity.md),
then merges them onto the African country–year trade-agreement panel produced by
[`01_clean_trade_agreements.ipynb`](01_clean_trade_agreements.ipynb).

**Why this is needed at all**, per the methodology document: the primary treatment
`Reciprocal_c,t` and the ordinal `depth_score` both take a *maximum* across a
country's Northern partners, which cannot distinguish (i) an agreement with an
economically massive partner (the EU) from one with a negligible partner (Malta
alone), or (ii) the true cumulative *breadth* of a country's Northern relationships
from a naive partner *count* — which, as the previous notebook's README caveat
documents, is structurally inflated because 23 of the 36 Northern partners are EU
member states coded as 23 identical bilateral rows. Weighting by real bilateral trade
value sidesteps the count-inflation problem entirely (each EU member still trades a
real, distinct amount), which is exactly why the methodology document recommends going
straight to trade-value weighting rather than patching the count.

**Data source: the CEPII Gravity dataset** (`data/cepii-gravity/`, see that folder's
README for full provenance and download instructions) — an origin–destination–year
panel with bilateral trade flows from several sources (IMF DOTS, UN Comtrade, BACI)
and GDP/population from the World Bank, IMF and Penn World Table.

**What this notebook does, in order:**

1. Load the small CEPII lookup/label files and sanity-check ISO3 coverage against the
   project's African/Northern lists (same two dicts as notebook 01).
2. Filter the (large) main Gravity CSV down to Africa↔Northern country-pairs, reading
   in chunks rather than loading the full ~4.7M-row file at once.
3. Investigate and correctly resolve a structural feature of the data: it is a
   *"squared" panel* (every country pair appears in every year of the full 1948–2021
   range, whether or not both countries existed yet), which silently creates duplicate
   rows for any Northern partner whose ISO3 code has been reused across a historical
   territorial change (e.g. Czechoslovakia → Czech Republic/Slovakia) unless handled
   correctly.
4. Choose and construct a bilateral trade-flow measure (IMF DOTS, mirror-averaged
   across the two reporting countries) and a bilateral GDP measure, and catch a subtle
   correctness bug along the way: a naive `.sum()` over an all-missing group in pandas
   silently returns `0`, not `NaN` — the wrong answer for a trade value where "zero
   trade" and "no data" are very different things.
5. Collapse to one row per (African country, Northern partner, year), and cross-check
   the resulting panel's coverage against the existing Baier–Bergstrand bilateral file.
6. Merge in the `reciprocal` classification from notebook 01 and build the
   country–year trade- and GDP-weighted exposure shares.
7. Validate the result against a known real-world case (South Africa's 2000 EU TDCA
   entry) before saving.
8. Merge onto the country–year panel from notebook 01 and save the enriched file.

## 1. Setup

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 140)

In [2]:
REPO_ROOT = Path("..")
GRAVITY_CSV = REPO_ROOT / "data" / "cepii-gravity" / "Gravity_V202211.csv"
BB_BILATERAL = REPO_ROOT / "data" / "processed" / "baier_bergstrand_africa_northern_bilateral.csv"
BB_COUNTRY_YEAR = REPO_ROOT / "data" / "processed" / "baier_bergstrand_africa_northern_country_year.csv"
OUT_DIR = REPO_ROOT / "data" / "processed"
OUT_DIR.mkdir(parents=True, exist_ok=True)

GRAVITY_CSV

WindowsPath('../data/cepii-gravity/Gravity_V202211.csv')

## 2. Country classifications and ISO3 sanity check

Same two fixed-vintage lists as notebook 01, keyed by ISO3 — CEPII's own ISO3 coding
turns out to match exactly, so no name-matching or crosswalk is needed here either.

In [3]:
AFRICAN = {
    "DZA": "Algeria", "AGO": "Angola", "BWA": "Botswana", "BFA": "Burkina Faso",
    "CMR": "Cameroon", "EGY": "Egypt", "SWZ": "Eswatini", "ETH": "Ethiopia",
    "GHA": "Ghana", "KEN": "Kenya", "LSO": "Lesotho", "MWI": "Malawi",
    "MUS": "Mauritius", "MAR": "Morocco", "MOZ": "Mozambique", "NAM": "Namibia",
    "NGA": "Nigeria", "RWA": "Rwanda", "SEN": "Senegal", "SLE": "Sierra Leone",
    "ZAF": "South Africa", "TZA": "Tanzania", "UGA": "Uganda", "ZMB": "Zambia",
}

NORTHERN = {
    "AUS": "Australia", "AUT": "Austria", "BEL": "Belgium", "CAN": "Canada",
    "CYP": "Cyprus", "CZE": "Czech Republic", "DNK": "Denmark", "EST": "Estonia",
    "FIN": "Finland", "FRA": "France", "DEU": "Germany", "GRC": "Greece",
    "HKG": "Hong Kong SAR", "ISL": "Iceland", "IRL": "Ireland", "ISR": "Israel",
    "ITA": "Italy", "JPN": "Japan", "KOR": "Republic of Korea", "LVA": "Latvia",
    "LTU": "Lithuania", "LUX": "Luxembourg", "MLT": "Malta", "NLD": "Netherlands",
    "NZL": "New Zealand", "NOR": "Norway", "PRT": "Portugal", "SGP": "Singapore",
    "SVK": "Slovakia", "SVN": "Slovenia", "ESP": "Spain", "SWE": "Sweden",
    "CHE": "Switzerland", "TWN": "Taiwan", "GBR": "United Kingdom", "USA": "United States",
}

countries = pd.read_csv(REPO_ROOT / "data" / "cepii-gravity" / "Countries_V202211.csv")
present = set(countries["iso3"])
missing = [c for c in {**AFRICAN, **NORTHERN} if c not in present]
print(f"{len(AFRICAN)} African countries, {len(NORTHERN)} Northern partners")
print(f"ISO3 codes not found in CEPII's Countries file (should be empty): {missing}")

24 African countries, 36 Northern partners
ISO3 codes not found in CEPII's Countries file (should be empty): []


## 3. Filter the main Gravity CSV to Africa↔Northern pairs

The full file is ~4.7M rows (all `country_id` pairs × 1948–2021), so it is read in
chunks with only the columns this notebook actually needs, rather than loaded whole.

In [4]:
USECOLS = [
    "year", "iso3_o", "iso3_d", "country_exists_o", "country_exists_d",
    "gdp_o", "gdp_d", "gdp_ppp_o", "gdp_ppp_d",
    "tradeflow_imf_o", "tradeflow_imf_d",
]

chunks = []
n_scanned = 0
for chunk in pd.read_csv(GRAVITY_CSV, usecols=USECOLS, chunksize=250_000):
    n_scanned += len(chunk)
    af_o, no_d = chunk["iso3_o"].isin(AFRICAN), chunk["iso3_d"].isin(NORTHERN)
    no_o, af_d = chunk["iso3_o"].isin(NORTHERN), chunk["iso3_d"].isin(AFRICAN)
    keep = chunk.loc[(af_o & no_d) | (no_o & af_d)]
    if len(keep):
        chunks.append(keep.copy())

raw = pd.concat(chunks, ignore_index=True)
print(f"Scanned {n_scanned:,} rows, kept {len(raw):,} Africa<->Northern rows "
      f"(both directions), years {raw['year'].min()}-{raw['year'].max()}")
raw.head()

Scanned 4,699,296 rows, kept 136,900 Africa<->Northern rows (both directions), years 1948-2021


,year,iso3_o,iso3_d,country_exists_o,country_exists_d,gdp_o,gdp_d,gdp_ppp_o,gdp_ppp_d,tradeflow_imf_o,tradeflow_imf_d
0,1948,AGO,AUS,1,1,NaN,NaN,NaN,NaN,NaN,NaN
1,1949,AGO,AUS,1,1,NaN,6608000.000,NaN,NaN,NaN,NaN
2,1950,AGO,AUS,1,1,NaN,7041999.872,NaN,NaN,NaN,NaN
3,1951,AGO,AUS,1,1,NaN,7457999.872,NaN,NaN,NaN,NaN
4,1952,AGO,AUS,1,1,NaN,8007000.064,NaN,NaN,NaN,NaN


## 4. A structural wrinkle: this is a "squared" panel

CEPII's Gravity dataset gives every `country_id` a row for *every* year in the full
1948–2021 range, whether or not that country existed yet that year — existence is
recorded separately via `country_exists_o`/`country_exists_d`, not by omitting rows.
That is fine on its own, but it interacts badly with ISO3 codes that have been reused
across a historical territorial change: Czechoslovakia's split, for instance, means
`iso3_o == "CZE"` and `iso3_o == "SVK"` **each** get a full 1948–2021 row set, so any
pair involving one of these Northern partners shows up more than once per year unless
the existence flags are used to pick the one row that actually applies.

In [5]:
dupe_check = raw.groupby(["iso3_o", "iso3_d"]).size()
print("Row count per (iso3_o, iso3_d) pair -- should be a single value (74 years) "
      "if there were no reused-code duplication:")
print(dupe_check.describe())
print()
worst = dupe_check.idxmax()
print(f"Worst offender: {worst} with {dupe_check.max()} rows (expected 74)")
raw.loc[(raw["iso3_o"] == worst[0]) & (raw["iso3_d"] == worst[1]),
        ["year", "country_exists_o", "country_exists_d"]].sort_values("year").head(8)

Row count per (iso3_o, iso3_d) pair -- should be a single value (74 years) if there were no reused-code duplication:
count    1728.000000
mean       79.224537
std        19.939353
min        74.000000
25%        74.000000
50%        74.000000
75%        74.000000
max       296.000000
dtype: float64

Worst offender: ('DEU', 'ETH') with 296 rows (expected 74)


,year,country_exists_o,country_exists_d
24420,1948,1,0
26196,1948,0,1
26270,1948,0,0
24346,1948,1,1
24347,1949,1,1
26271,1949,0,0
26197,1949,0,1
24421,1949,1,0


Confirmed: for any given year, exactly one of the candidate rows has
`country_exists_o == 1 AND country_exists_d == 1` — filtering on both flags together
(not just one) resolves the duplication cleanly, exactly the way notebook 01 used
Baier–Bergstrand's own `NoCty` flag to distinguish "no agreement" from "not yet a
country."   This also naturally restricts the sample to the project's 1950–2017
window.

In [6]:
raw = raw[(raw["country_exists_o"] == 1) & (raw["country_exists_d"] == 1)].copy()
raw = raw[(raw["year"] >= 1950) & (raw["year"] <= 2017)].copy()

dupes_left = raw.duplicated(subset=["year", "iso3_o", "iso3_d"]).sum()
print(f"{len(raw):,} rows after the existence-flag filter; "
      f"duplicate (year, iso3_o, iso3_d) combinations remaining: {dupes_left}")
print(f"Distinct ordered pairs covered: {raw[['iso3_o','iso3_d']].drop_duplicates().shape[0]} "
      f"of {24*36*2} possible")

104,784 rows after the existence-flag filter; duplicate (year, iso3_o, iso3_d) combinations remaining: 0
Distinct ordered pairs covered: 1728 of 1728 possible


## 5. Choose the trade-flow source, and mirror-average it

Three bilateral trade sources are bundled in CEPII Gravity: **IMF DOTS** (deepest
historical coverage, back to 1948), **UN Comtrade** (from 1962), and **BACI**
(reconciled/highest-quality, but only from 1996). Given this project's 1950–2017
window — and given that the level-1 GSP/AGOA-era relationships the methodology
document emphasizes run from 1976 well before BACI's 1996 start — **IMF DOTS is used
as the primary source**, checked below for adequate coverage.

Each directional row carries *two* IMF figures for the same underlying flow —
`tradeflow_imf_o` (as reported by the origin country) and `tradeflow_imf_d` (the same
flow as reported by the destination, a "mirror" statistic) — because bilateral trade
statistics from two national sources routinely disagree. Averaging the two where both
are available is standard gravity-literature practice for exactly this reason.

In [7]:
raw["flow_imf"] = raw[["tradeflow_imf_o", "tradeflow_imf_d"]].mean(axis=1, skipna=True)

w = raw
for c in ["tradeflow_imf_o", "tradeflow_imf_d", "flow_imf", "gdp_o", "gdp_d"]:
    print(f"{c:16s} non-null share: {w[c].notna().mean():.1%}")

tradeflow_imf_o  non-null share: 61.5%
tradeflow_imf_d  non-null share: 63.9%
flow_imf         non-null share: 72.3%
gdp_o            non-null share: 90.2%
gdp_d            non-null share: 90.2%


## 6. Collapse to one row per (African country, Northern partner, year)

Trade is directional (African country's exports to the Northern partner, and the
reverse), so the two directional rows are **summed** to get total bilateral trade —
unlike notebook 01's agreement-*level* variable, which was collapsed by *maximum*
because non-reciprocal preference codes are directional in a different sense (which
side grants the preference, not a flow to be added up). GDP is a property of the
Northern partner itself, not a flow, so it is simply carried over rather than summed.

**A pandas pitfall worth catching explicitly**: `Series.sum()` on a group where every
value is `NaN` returns `0.0`, not `NaN` — silently turning "no trade data available
this year" into "zero trade happened," which would badly bias any weighted-average
built on top of it. `min_count=1` fixes this for a true missing-data case; but see
§8 below for a second, easy-to-miss case of the same underlying issue.

In [8]:
raw["african_iso3"] = np.where(raw["iso3_o"].isin(AFRICAN), raw["iso3_o"], raw["iso3_d"])
raw["northern_iso3"] = np.where(raw["iso3_o"].isin(AFRICAN), raw["iso3_d"], raw["iso3_o"])
africa_is_o = raw["iso3_o"].isin(AFRICAN)
raw["gdp_northern"] = np.where(africa_is_o, raw["gdp_d"], raw["gdp_o"])
raw["gdp_ppp_northern"] = np.where(africa_is_o, raw["gdp_ppp_d"], raw["gdp_ppp_o"])


def sum_or_nan(s):
    # Plain .sum() on an all-NaN group wrongly returns 0 -- return NaN instead.
    return s.sum() if s.notna().any() else np.nan


bilateral = raw.groupby(["african_iso3", "northern_iso3", "year"], as_index=False).agg(
    trade_imf=("flow_imf", sum_or_nan),
    trade_imf_n_directions=("flow_imf", "count"),
    gdp_northern=("gdp_northern", "mean"),
    gdp_ppp_northern=("gdp_ppp_northern", "mean"),
)

print(f"{len(bilateral):,} bilateral (African, Northern, year) rows "
      f"(expect up to {24*36*68:,})")
print(f"trade_imf non-null share: {bilateral['trade_imf'].notna().mean():.1%}")
bilateral.head()

52,392 bilateral (African, Northern, year) rows (expect up to 58,752)
trade_imf non-null share: 77.6%


,african_iso3,northern_iso3,year,trade_imf,trade_imf_n_directions,gdp_northern,gdp_ppp_northern
0,AGO,AUS,1950,NaN,0,7041999.872,NaN
1,AGO,AUS,1951,NaN,0,7457999.872,NaN
2,AGO,AUS,1952,NaN,0,8007000.064,NaN
3,AGO,AUS,1953,NaN,0,9137000.448,NaN
4,AGO,AUS,1954,NaN,0,9966000.128,NaN


## 7. Cross-check against the existing Baier–Bergstrand bilateral panel

`01_clean_trade_agreements.ipynb` already built a bilateral (African, Northern, year)
panel from the agreement-level data with its own existence coding — do the two
sources agree on which country-pair-years exist?

In [9]:
bb_bilateral = pd.read_csv(BB_BILATERAL)
print(f"Baier-Bergstrand bilateral rows: {len(bb_bilateral):,}")

merged_check = bb_bilateral.merge(
    bilateral, on=["african_iso3", "northern_iso3", "year"], how="left", indicator=True
)
print(merged_check["_merge"].value_counts())
print()

missing = merged_check[merged_check["_merge"] == "left_only"]
print("Northern partners behind the gap, by first/last missing year:")
print(missing.groupby("northern_iso3")["year"].agg(["min", "max", "count"]))

Baier-Bergstrand bilateral rows: 58,752
_merge
both          52392
left_only      6360
right_only        0
Name: count, dtype: int64

Northern partners behind the gap, by first/last missing year:
                min   max  count
northern_iso3                   
CZE            1950  1992   1032
EST            1950  1990    984
LTU            1950  1990    984
LVA            1950  1990    984
SGP            1950  1964    360
SVK            1950  1992   1032
SVN            1950  1990    984


**The gap is fully explained, and it is CEPII being *more* correct, not less.**
Every affected partner is a country that did not yet exist as an independent state in
the missing years: Czechoslovakia's 1993 split (`CZE`/`SVK`), the Baltic states'
1991 independence from the USSR (`EST`/`LVA`/`LTU`), Slovenia's 1991 independence from
Yugoslavia (`SVN`), and Singapore's 1965 independence (`SGP`). CEPII's
`country_exists` flag is simply stricter about pre-independence years than
Baier–Bergstrand's own coding is for these specific cases — a legitimate definitional
difference between the two sources, not a data error. It only affects the *weighting*
variables built below, for a handful of Northern partners in years before those
partners existed, and only narrows an already-large denominator.

## 8. Merge in `reciprocal` and build the country–year exposure shares

For each African country–year, the **trade-weighted exposure** is the share of its
total bilateral trade with Northern partners that sits with a partner at reciprocal
(level ≥ 2) status; the **GDP-weighted exposure** is the analogous share of Northern
partners' combined GDP. Both are continuous variables in `[0, 1]`, in contrast to the
binary `Reciprocal_c,t` treatment.

**The same all-NaN-sum pitfall from §6 reappears here in a different, easy-to-miss
form.** When a country has *zero* reciprocal partners in a given year, the correct
numerator is a genuine `0` (there is no trade happening under reciprocal terms,
because no such partner relationship exists) — not `NaN`. But when the *denominator*
(total Northern trade) is entirely missing, that must stay `NaN` (unknown), not
silently become `0`. The two cases need opposite `min_count` handling, and conflating
them was the actual bug caught while building this notebook (see the file history for
the earlier, wrong version): treating "zero reciprocal partners" as missing data
dropped 86% of country-years from the trade-weighted share for no real reason.

In [10]:
bb_country_year = pd.read_csv(BB_COUNTRY_YEAR)

m = bb_bilateral.merge(bilateral, on=["african_iso3", "northern_iso3", "year"], how="left")
m["reciprocal_partner"] = np.where(m["level"].notna(), (m["level"] >= 2).astype(float), np.nan)


def _exposure(g):
    return pd.Series({
        # denominator: NaN if literally no partner has trade/GDP data that year
        "trade_total_northern": g["trade_imf"].sum(min_count=1),
        "gdp_total_northern": g["gdp_northern"].sum(min_count=1),
        # numerator: correctly 0 (not NaN) when zero partners are at reciprocal level
        # that year -- default (no min_count) sums an empty/all-NaN selection to 0.
        "trade_reciprocal_northern": g.loc[g["reciprocal_partner"] == 1, "trade_imf"].sum(),
        "gdp_reciprocal_northern": g.loc[g["reciprocal_partner"] == 1, "gdp_northern"].sum(),
        "n_reciprocal_partners": int((g["reciprocal_partner"] == 1).sum()),
        "n_partners_trade_available": int(g["trade_imf"].notna().sum()),
    })


exposure = (
    m.groupby(["african_iso3", "year"]).apply(_exposure, include_groups=False).reset_index()
)
exposure["reciprocal_trade_share"] = (
    exposure["trade_reciprocal_northern"] / exposure["trade_total_northern"]
)
exposure["reciprocal_gdp_share"] = (
    exposure["gdp_reciprocal_northern"] / exposure["gdp_total_northern"]
)

print(f"reciprocal_trade_share non-null: {exposure['reciprocal_trade_share'].notna().mean():.1%}")
print(f"reciprocal_gdp_share non-null:   {exposure['reciprocal_gdp_share'].notna().mean():.1%}")
print(f"Bounds -- trade share: [{exposure['reciprocal_trade_share'].min():.3f}, "
      f"{exposure['reciprocal_trade_share'].max():.3f}], "
      f"GDP share: [{exposure['reciprocal_gdp_share'].min():.3f}, "
      f"{exposure['reciprocal_gdp_share'].max():.3f}]")

reciprocal_trade_share non-null: 91.4%
reciprocal_gdp_share non-null:   100.0%
Bounds -- trade share: [0.000, 0.945], GDP share: [0.000, 0.783]


## 9. Validate against a known case: South Africa's 2000 EU TDCA entry

South Africa's reciprocal EU Trade, Development and Cooperation Agreement entered
into force in 2000 — `reciprocal` should jump from 0 to 1 exactly then, and, since the
EU is South Africa's dominant Northern trading partner, `reciprocal_trade_share`
should jump sharply at the same point rather than drift gradually.

In [11]:
check = bb_country_year.merge(
    exposure, left_on=["iso3", "year"], right_on=["african_iso3", "year"], how="left"
)
zaf = check[check["iso3"] == "ZAF"].sort_values("year")
zaf.loc[zaf["year"].between(1997, 2003),
        ["year", "depth_score", "reciprocal", "reciprocal_trade_share", "reciprocal_gdp_share"]]

,year,depth_score,reciprocal,reciprocal_trade_share,reciprocal_gdp_share
1543,1997,1.0,0.0,0.000000,0.000000
1544,1998,1.0,0.0,0.000000,0.000000
1545,1999,1.0,0.0,0.000000,0.000000
1546,2000,3.0,1.0,0.554172,0.316018
1547,2001,3.0,1.0,0.575031,0.321571
1548,2002,3.0,1.0,0.587830,0.335812
1549,2003,3.0,1.0,0.575587,0.365326


Exactly the expected pattern: `reciprocal_trade_share` sits at `0.000` through
1999, then jumps to `0.55` the same year `reciprocal` switches to `1` — over half of
South Africa's recorded Northern-partner trade sits with the EU, consistent with it
being by far South Africa's largest Northern trading relationship. This is a real,
concrete confirmation that the weighting logic behaves sensibly, not just that it runs
without error.

For contrast, `mean(reciprocal_trade_share | reciprocal==0)` across the whole panel
should be exactly `0`, and `mean(... | reciprocal==1)` should be well above `0` but
below `1` (Northern trade includes non-reciprocal partners too, even once at least one
reciprocal partner exists):

In [12]:
check.groupby("reciprocal")[["reciprocal_trade_share", "reciprocal_gdp_share"]].mean()

,reciprocal_trade_share,reciprocal_gdp_share
reciprocal,,
0.0,0.000000,0.000000
1.0,0.614171,0.388653


## 10. Save the outputs

Two files: the bilateral CEPII panel (parallel to notebook 01's bilateral file, for
future partner-level extensions), and the enriched country–year panel — notebook 01's
file with the two new exposure columns merged on.

In [13]:
bilateral_out = OUT_DIR / "cepii_gravity_africa_northern_bilateral.csv"
bilateral.to_csv(bilateral_out, index=False)
print(f"Saved {len(bilateral):,} rows -> {bilateral_out}")

enriched = bb_country_year.merge(
    exposure.rename(columns={"african_iso3": "iso3"}), on=["iso3", "year"], how="left"
)

enriched_out = OUT_DIR / "trade_agreements_with_exposure_country_year.csv"
enriched.to_csv(enriched_out, index=False)
print(f"Saved {len(enriched):,} rows -> {enriched_out}")
enriched.head()

Saved 52,392 rows -> ..\data\processed\cepii_gravity_africa_northern_bilateral.csv
Saved 1,632 rows -> ..\data\processed\trade_agreements_with_exposure_country_year.csv


,iso3,year,country_exists,depth_score,agreement_type,reciprocal,fta_or_deeper,nonreciprocal_only,n_northern_reporting,n_northern_with_agreement,trade_total_northern,gdp_total_northern,trade_reciprocal_northern,gdp_reciprocal_northern,n_reciprocal_partners,n_partners_trade_available,reciprocal_trade_share,reciprocal_gdp_share
0,AGO,1950,True,0.0,0_none,0.0,0.0,0.0,28.0,0.0,109650.0,4.391497e+08,0.0,0.0,0.0,14.0,0.0,0.0
1,AGO,1951,True,0.0,0_none,0.0,0.0,0.0,28.0,0.0,152450.0,5.087046e+08,0.0,0.0,0.0,15.0,0.0,0.0
2,AGO,1952,True,0.0,0_none,0.0,0.0,0.0,28.0,0.0,154550.0,5.641776e+08,0.0,0.0,0.0,19.0,0.0,0.0
3,AGO,1953,True,0.0,0_none,0.0,0.0,0.0,28.0,0.0,179100.0,5.948022e+08,0.0,0.0,0.0,18.0,0.0,0.0
4,AGO,1954,True,0.0,0_none,0.0,0.0,0.0,28.0,0.0,183750.0,6.089962e+08,0.0,0.0,0.0,17.0,0.0,0.0


## 11. Limitations and next steps

- **IMF DOTS is used rather than BACI or Comtrade** because of its deeper historical
  coverage (from 1948, vs. 1962 and 1996 respectively) — a deliberate choice for this
  project's 1950–2017 window, not a default; Comtrade/BACI remain available in
  `data/cepii-gravity/` for a robustness cross-check on the post-1996/1962 subsample if
  needed later.
- **The CZE/SVK/Baltic-states/Slovenia/Singapore existence-flag gap** documented in
  §7 only affects the weighting denominators for those specific pre-independence
  years; it does not affect `Reciprocal_c,t` or `depth_score`, which come entirely
  from notebook 01.
- **`reciprocal_trade_share` has ~9% missing country-years** (vs. 0% for the
  GDP-weighted version) — years where no Northern partner has IMF trade data at all,
  concentrated in the earliest part of the sample. Prefer `reciprocal_gdp_share` where
  full-sample coverage matters more than trade-value precision.
- **Next step**: merge `trade_agreements_with_exposure_country_year.csv` onto the GGDC
  sectoral productivity panel (`data/Global-Productivity-Sectoral-Database.dta`) on
  `iso3`/country and `year`, per §6.2 step 6 of the methodology document, then run the
  McMillan–Rodrik decomposition (§6.3).